# 6. Results Summary & Discussion

Cross-store comparison of the three forecasting methods, and the ethical/social/policy discussion for the paper. Corresponds to step 6 of the workflow in `CLAUDE.md`.

In [ ]:
import sys
from pathlib import Path

SRC = Path.cwd().parent / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from plotting import set_paper_style, METHOD_COLORS, METHOD_LABELS, METHOD_ORDER

set_paper_style()

OUTPUTS = Path.cwd().parent / "outputs"
results = pd.read_csv(OUTPUTS / "tables" / "model_comparison.csv")
store_meta = pd.read_csv(OUTPUTS / "tables" / "store_selection.csv")

test_results = (
    results.query("split == 'test'")
    .merge(store_meta, left_on="store", right_on="Store")
)
test_results

## RMSE by store and method

In [ ]:
stores = sorted(test_results["store"].unique())
x = np.arange(len(stores))
width = 0.25

fig, ax = plt.subplots(figsize=(8, 4))
for i, method in enumerate(METHOD_ORDER):
    vals = [
        test_results.query("store == @s and method == @method")["rmse"].iloc[0]
        for s in stores
    ]
    ax.bar(x + (i - 1) * width, vals, width, label=METHOD_LABELS[method], color=METHOD_COLORS[method])

ax.set_xticks(x)
ax.set_xticklabels([f"Store {s}" for s in stores])
ax.set_ylabel("Test RMSE (sales)")
ax.set_title("Forecast error by method and store")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(OUTPUTS / "figures" / "rmse_comparison.png")
plt.show()

## Best method per store

In [ ]:
best = (
    test_results.sort_values("rmse")
    .groupby("store")
    .first()
    [["method", "rmse", "mae", "StoreType", "Promo2", "DistanceTier"]]
)
best

## Discussion: ethical, social, and policy implications

Store-level sales forecasts like these are typically consumed downstream by staffing and inventory-ordering systems: a store predicted to have low demand next week may be scheduled fewer staff-hours, and a store predicted to have high demand may receive priority stock allocation. This creates a direct link between forecast *error* and real workplace outcomes - a store whose sales the model under-predicts risks being understaffed relative to actual footfall (worse customer service, more pressure on whoever is on shift), while over-prediction ties up inventory capital and can lead to wasted stock (particularly relevant for perishable assortments).

The `best`-method table above shows whether forecast accuracy is uniform across store characteristics or varies systematically with StoreType, Promo2, or CompetitionDistance - fill in the actual pattern found here once the notebook has been run. If a retailer used a single shared model (or a single method) across all stores, any such unevenness would mean some store types - and by extension the workers and communities they serve - systematically receive worse-calibrated staffing and stock decisions than others. That is a fairness question distinct from raw average accuracy: even a model with good aggregate performance can disadvantage a subset of stores (e.g. smaller or more rural locations, or a StoreType with sparser training data) in a way that compounds over time.

A responsible deployment would monitor forecast error *by store segment*, not just in aggregate, and treat a persistently under-served segment as a signal to revisit the model - or fall back to a simpler, more conservative baseline for those stores - rather than silently accepting worse service for them.